# Handcrafted Image Retrieval Pipeline — GPU Accelerated
## Oxford & Paris Buildings

| Stage | Component | Acceleration |
|-------|-----------|-------------|
| 🔴 Offline | RootSIFT extraction | CPU (SIFT has no stable GPU impl) |
| 🔴 Offline | K-Means vocabulary | **FAISS GPU** |
| 🔴 Offline | VLAD nearest-centroid assignment | **FAISS GPU** |
| 🔴 Offline | VLAD matrix ops | **CuPy** |
| 🔴 Offline | PCA + Whitening | **cuML GPU** (fallback: sklearn CPU) |
| 🟢 Online  | Cosine similarity retrieval | **FAISS GPU IndexFlatIP** |
| 🟢 Online  | RANSAC spatial re-ranking | CPU (OpenCV) |
| 🟢 Online  | AQE | **CuPy** |

---
## Cell 1 — Install & Import

In [3]:
!pip install faiss-cpu

In [4]:
import os, sys, time, pickle, warnings, tarfile, urllib.request
warnings.filterwarnings('ignore')

import numpy as np
import cv2
from pathlib import Path
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

# ── GPU libraries ──────────────────────────────────────────────────────────
import faiss
print(f'FAISS version : {faiss.__version__}')

try:
    import cupy as cp
    cp.cuda.Device(0).use()
    USE_CUPY = True
    print(f'CuPy version  : {cp.__version__}')
    print(f'GPU memory    : {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total')
except Exception as e:
    USE_CUPY = False
    print(f'CuPy not available ({e}) — using NumPy for array ops')

try:
    from cuml.decomposition import PCA as cuPCA
    USE_CUML = True
    print('cuML PCA      : available')
except Exception:
    USE_CUML = False
    print('cuML PCA      : not available — using sklearn CPU')

# Check FAISS GPU resources
N_GPUS = faiss.get_num_gpus()
print(f'FAISS GPUs    : {N_GPUS}')
USE_FAISS_GPU = N_GPUS > 0
if USE_FAISS_GPU:
    GPU_RES = faiss.StandardGpuResources()
    print('FAISS GPU resources initialised')
else:
    print('FAISS running on CPU')

FAISS version : 1.13.2
CuPy version  : 14.0.1
GPU memory    : 15.6 GB total
cuML PCA      : available
FAISS GPUs    : 0
FAISS running on CPU


---
## Cell 2 — Configuration & GT Download

In [5]:
DATASET_ROOT = Path('/kaggle/input/datasets/jeffreyamc/oxford-paris-buildings-v2')
OXFORD_IMGS  = DATASET_ROOT / 'oxford'
PARIS_IMGS   = DATASET_ROOT / 'paris'

OUTPUT_DIR = Path('/kaggle/working/retrieval_output')
CACHE_DIR  = OUTPUT_DIR / 'cache'
GT_DIR     = OUTPUT_DIR / 'ground_truth'
for d in [OUTPUT_DIR, CACHE_DIR, GT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATASETS_TO_USE = ['oxford', 'paris']

# ── Hyperparameters ────────────────────────────────────────────────────────
K_VLAD          = 1024
PCA_DIM         = 256
SIFT_N_FEATURES = 0
MAX_IMAGE_SIZE  = 1024
N_WORKERS       = 4      # CPU threads for SIFT extraction
RERANK_TOP_N    = 100    # RANSAC candidates
MIN_INLIERS     = 6
AQE_TOP_K       = 10

# ── Landmark sets (those with official GT) ────────────────────────────────
OXFORD_GT_LANDMARKS = {
    'all_souls','ashmolean','balliol','bodleian','christ_church',
    'cornmarket','hertford','keble','magdalen','pitt_rivers','radcliffe_camera'
}
PARIS_GT_LANDMARKS = {
    'defense','eiffel','general','invalides','louvre','moulinrouge',
    'museedorsay','notredame','pantheon','pompidou','sacrecoeur','triomphe'
}

GT_URLS = {
    'oxford': 'https://www.robots.ox.ac.uk/~vgg/data/oxbuildings/gt_files_170407.tgz',
    'paris' : 'https://www.robots.ox.ac.uk/~vgg/data/parisbuildings/paris_120310.tgz',
}

def download_gt(dataset_name, url, gt_dir):
    marker = gt_dir / f'.{dataset_name}_downloaded'
    if marker.exists():
        print(f'[GT] {dataset_name} already downloaded')
        return
    tgz = gt_dir / f'{dataset_name}_gt.tgz'
    print(f'[GT] Downloading {dataset_name}...')
    urllib.request.urlretrieve(url, tgz)
    with tarfile.open(tgz) as t: t.extractall(gt_dir)
    tgz.unlink(); marker.touch()
    print(f'     Done')

for ds in DATASETS_TO_USE:
    download_gt(ds, GT_URLS[ds], GT_DIR)

print(f'GT query files: {len(list(GT_DIR.rglob("*_query.txt")))}')

[GT] oxford already downloaded
[GT] paris already downloaded
GT query files: 110


---
## Cell 3 — Dataset Scanner

In [6]:
def find_gt_dir(gt_root):
    files = list(gt_root.rglob('*_query.txt'))
    return files[0].parent if files else None

def scan_dataset(img_root, gt_root, dataset_name, gt_landmarks):
    img_paths, stem2path, queries = [], {}, []
    if not img_root.exists():
        print(f'[WARN] {img_root} not found'); return img_paths, stem2path, queries

    lm_dirs = sorted(d for d in img_root.iterdir() if d.is_dir())
    print(f'\n{"="*60}\n  {dataset_name.upper()} — {len(lm_dirs)} folders\n{"="*60}')
    n_dist = 0
    for lm in lm_dirs:
        is_gt = lm.name in gt_landmarks
        imgs  = sorted(lm.glob('*.jpg')) + sorted(lm.glob('*.JPG')) + sorted(lm.glob('*.png'))
        for p in imgs:
            img_paths.append(p); stem2path[p.stem] = p
        if not is_gt: n_dist += len(imgs)
        print(f'  [{"GT landmark" if is_gt else "DISTRACTOR "}]  {lm.name:20s}: {len(imgs):5d} imgs')

    gt_flat = find_gt_dir(gt_root)
    if gt_flat is None:
        print('[WARN] No GT files found'); return img_paths, stem2path, queries

    def clean(s):
        if dataset_name == 'oxford':
            return s.replace('oxc1_', '').strip()   # solo quitar oxc1_
        else:  # paris
            return s.strip()

    def read_set(gt_flat, stem, sfx):
        p = gt_flat / f'{stem}_{sfx}.txt'
        if not p.exists(): return set()
        with open(p) as f: return {clean(l) for l in f if l.strip()}

    for qf in sorted(gt_flat.glob('*_query.txt')):
        qs = qf.stem.replace('_query','')
        with open(qf) as f: parts = f.read().strip().split()
        queries.append({
            'name'      : f'{dataset_name}_{qs}',
            'dataset'   : dataset_name,
            'query_img' : clean(parts[0]),
            'query_roi' : list(map(float, parts[1:5])) if len(parts)>=5 else None,
            'good'      : read_set(gt_flat, qs, 'good'),
            'ok'        : read_set(gt_flat, qs, 'ok'),
            'junk'      : read_set(gt_flat, qs, 'junk'),
        })

    print(f'  Total: {len(img_paths)} images ({n_dist} distractors), {len(queries)} queries')
    return img_paths, stem2path, queries


ALL_IMG_PATHS, STEM2PATH, ALL_QUERIES = [], {}, []
if 'oxford' in DATASETS_TO_USE:
    a,b,c = scan_dataset(OXFORD_IMGS, GT_DIR, 'oxford', OXFORD_GT_LANDMARKS)
    ALL_IMG_PATHS+=a; STEM2PATH.update(b); ALL_QUERIES+=c
if 'paris' in DATASETS_TO_USE:
    a,b,c = scan_dataset(PARIS_IMGS,  GT_DIR, 'paris',  PARIS_GT_LANDMARKS)
    ALL_IMG_PATHS+=a; STEM2PATH.update(b); ALL_QUERIES+=c

N_IMAGES    = len(ALL_IMG_PATHS)
DB_STEMS    = [p.stem for p in ALL_IMG_PATHS]
STEM_TO_IDX = {s:i for i,s in enumerate(DB_STEMS)}
oxford_queries = [q for q in ALL_QUERIES if q['dataset']=='oxford']
paris_queries  = [q for q in ALL_QUERIES if q['dataset']=='paris']
print(f'\nDB={N_IMAGES}  Oxford_q={len(oxford_queries)}  Paris_q={len(paris_queries)}')


  OXFORD — 17 folders
  [GT landmark]  all_souls           :   132 imgs
  [GT landmark]  ashmolean           :   195 imgs
  [GT landmark]  balliol             :   154 imgs
  [GT landmark]  bodleian            :   214 imgs
  [GT landmark]  christ_church       :   543 imgs
  [GT landmark]  cornmarket          :    59 imgs
  [GT landmark]  hertford            :    67 imgs
  [DISTRACTOR ]  jesus               :   161 imgs
  [GT landmark]  keble               :   121 imgs
  [GT landmark]  magdalen            :   685 imgs
  [DISTRACTOR ]  new                 :   458 imgs
  [DISTRACTOR ]  oriel               :    95 imgs
  [DISTRACTOR ]  oxford              :  1502 imgs
  [GT landmark]  pitt_rivers         :   108 imgs
  [GT landmark]  radcliffe_camera    :   282 imgs
  [DISTRACTOR ]  trinity             :   217 imgs
  [DISTRACTOR ]  worcester           :    70 imgs
  Total: 5063 images (2503 distractors), 110 queries

  PARIS — 12 folders
  [GT landmark]  defense             :   381 imgs
  

---
# 🔴 OFFLINE STAGE
## Offline Step 1 — RootSIFT (CPU)
> SIFT detection runs on CPU — no stable GPU implementation in OpenCV.  
> We parallelise with `ThreadPoolExecutor` to use all CPU cores.

In [7]:
def compute_rootsift(descs):
    descs = descs.astype(np.float32)
    descs /= (descs.sum(axis=1, keepdims=True) + 1e-7)
    return np.sqrt(descs)

def extract_rootsift(image_path, roi=None,
                     max_size=MAX_IMAGE_SIZE, n_features=SIFT_N_FEATURES):
    try:
        img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
        if img is None: return [], None
        h, w  = img.shape
        scale = min(max_size / max(h, w), 1.0)
        if scale < 1.0:
            img = cv2.resize(img,(int(w*scale),int(h*scale)),interpolation=cv2.INTER_AREA)
        sift = cv2.SIFT_create(nfeatures=n_features,contrastThreshold=0.04,edgeThreshold=10)
        kps, descs = sift.detectAndCompute(img, None)
        if descs is None or len(descs)==0: return [], None
        descs = compute_rootsift(descs)
        if roi is not None:
            x1,y1,x2,y2 = [c*scale for c in roi]
            mask  = np.array([x1<=kp.pt[0]<=x2 and y1<=kp.pt[1]<=y2 for kp in kps],dtype=bool)
            kps   = [kp for kp,m in zip(kps,mask) if m]
            descs = descs[mask] if mask.any() else None
            if descs is None or len(descs)==0: return [], None
        return kps, descs
    except Exception as e:
        return [], None

# Test
kps, d = extract_rootsift(ALL_IMG_PATHS[0])
print(f'RootSIFT OK: {len(kps)} kps, norm={np.linalg.norm(d[0]):.4f}')

RootSIFT OK: 5260 kps, norm=1.0000


## Offline Step 2 — Visual Vocabulary with FAISS K-Means (GPU)
> `faiss.Kmeans` with `gpu=True` is **10-50× faster** than sklearn MiniBatchKMeans on T4.

In [8]:
VOCAB_CACHE = CACHE_DIR / f'vocab_K{K_VLAD}.npy'   # store centroids as .npy

def build_vocabulary_faiss(image_paths, k=K_VLAD, max_desc_per_img=150,
                            n_sample_imgs=2000, cache_path=None):
    """
    Build visual vocabulary using FAISS K-Means.
    Uses GPU if available (faiss.Kmeans gpu=True).
    Returns centroids as (K, 128) float32 numpy array.
    """
    if cache_path and cache_path.exists():
        print(f'[CACHE] Vocabulary ← {cache_path.name}')
        return np.load(cache_path)

    print(f'Building K={k} vocabulary  '
          f'[{"FAISS-GPU" if USE_FAISS_GPU else "FAISS-CPU"}]...')
    t0 = time.time()

    rng  = np.random.default_rng(42)
    idxs = rng.choice(len(image_paths), min(n_sample_imgs, len(image_paths)), replace=False)
    all_descs = []
    for i in tqdm(idxs, desc='Collecting descriptors'):
        _, d = extract_rootsift(image_paths[i])
        if d is not None and len(d) > 0:
            if len(d) > max_desc_per_img:
                d = d[np.random.choice(len(d), max_desc_per_img, replace=False)]
            all_descs.append(d)

    X = np.vstack(all_descs).astype(np.float32)
    print(f'  {len(X):,} descriptors → clustering into {k} words...')

    # FAISS K-Means — GPU accelerated
    kmeans = faiss.Kmeans(
        d=128,          # descriptor dimension
        k=k,
        niter=100,      # iterations (vs 300 for sklearn, but FAISS converges faster)
        nredo=3,        # multiple restarts
        verbose=True,
        gpu=USE_FAISS_GPU
    )
    kmeans.train(X)
    centroids = kmeans.centroids   # (K, 128) float32

    print(f'  Done in {time.time()-t0:.1f}s')
    if cache_path:
        np.save(cache_path, centroids)
        print(f'  Cached → {cache_path.name}')
    return centroids


CENTROIDS = build_vocabulary_faiss(ALL_IMG_PATHS, k=K_VLAD, cache_path=VOCAB_CACHE)
print(f'Centroids: {CENTROIDS.shape}')

[CACHE] Vocabulary ← vocab_K1024.npy
Centroids: (1024, 128)


## Offline Step 3 — VLAD Encoding with FAISS GPU Assignment
> The bottleneck inside VLAD is the **nearest centroid search** (N descriptors × K centroids).  
> We batch all descriptors per image and use a FAISS GPU index for assignment.

In [9]:
# ── Build a FAISS index over centroids for fast assignment ─────────────────
# IndexFlatL2 on GPU → exact nearest centroid in milliseconds
def build_centroid_index(centroids):
    index_cpu = faiss.IndexFlatL2(128)
    index_cpu.add(centroids)
    if USE_FAISS_GPU:
        index = faiss.index_cpu_to_gpu(GPU_RES, 0, index_cpu)
        print(f'Centroid index on GPU  ({len(centroids)} centroids)')
    else:
        index = index_cpu
        print(f'Centroid index on CPU  ({len(centroids)} centroids)')
    return index


CENTROID_INDEX = build_centroid_index(CENTROIDS)


def vlad_encode_faiss(descs: np.ndarray, centroids: np.ndarray,
                      centroid_index, power_norm=True) -> np.ndarray:
    """
    Encode descriptors as VLAD using FAISS GPU for centroid assignment.

    Speed vs CPU numpy:
        CPU: O(N*K) dot products per image
        GPU: FAISS batch search — orders of magnitude faster for large K
    """
    K, D = centroids.shape
    descs_f32 = np.ascontiguousarray(descs.astype(np.float32))

    # GPU nearest-centroid assignment
    _, assign = centroid_index.search(descs_f32, 1)   # (N, 1)
    assign    = assign.flatten()                        # (N,)

    # Residual accumulation (on CPU — small matrix K×128)
    vlad = np.zeros((K, D), dtype=np.float32)
    np.add.at(vlad, assign, descs_f32 - centroids[assign])

    if power_norm:
        vlad = np.sign(vlad) * np.sqrt(np.abs(vlad))

    vlad = vlad.flatten()
    n    = np.linalg.norm(vlad)
    return vlad / n if n > 1e-8 else vlad


def extract_vlad(path, centroids, centroid_index, roi=None):
    _, descs = extract_rootsift(path, roi=roi)
    if descs is None or len(descs) == 0:
        return np.zeros(len(centroids)*128, dtype=np.float32)
    return vlad_encode_faiss(descs, centroids, centroid_index)


# Test
v = extract_vlad(ALL_IMG_PATHS[0], CENTROIDS, CENTROID_INDEX)
print(f'VLAD shape: {v.shape}  norm={np.linalg.norm(v):.4f}')

Centroid index on CPU  (1024 centroids)
VLAD shape: (131072,)  norm=1.0000


## Offline Step 4 — Extract VLAD for ALL Database Images

In [10]:
VLAD_RAW_CACHE = CACHE_DIR / f'vlad_raw_K{K_VLAD}.npy'
STEMS_CACHE    = CACHE_DIR / 'db_stems.pkl'

def build_database(image_paths, centroids, centroid_index,
                   cache_vlad=None, cache_stems=None):
    if cache_vlad and cache_vlad.exists() and \
       cache_stems and cache_stems.exists():
        print(f'[CACHE] DB ← {cache_vlad.name}')
        vlad = np.load(cache_vlad)
        with open(cache_stems,'rb') as f: stems = pickle.load(f)
        print(f'  {len(stems)} vectors, dim={vlad.shape[1]}')
        return vlad, stems

    print(f'[OFFLINE] Extracting VLAD for {len(image_paths)} images...')
    t0  = time.time()
    dim = len(centroids) * 128
    mat = np.zeros((len(image_paths), dim), dtype=np.float32)
    stms = [''] * len(image_paths)

    # Note: FAISS centroid_index is NOT thread-safe when on GPU.
    # We extract SIFT in parallel (CPU-bound) but do VLAD encoding
    # (GPU centroid search) in the main thread in batches.
    print('  Phase 1: parallel SIFT extraction...')
    raw_descs = [None] * len(image_paths)

    def sift_worker(args):
        idx, path = args
        _, descs = extract_rootsift(path)
        return idx, path.stem, descs

    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = {ex.submit(sift_worker,(i,p)):i for i,p in enumerate(image_paths)}
        for fut in tqdm(as_completed(futs), total=len(futs), desc='SIFT (CPU)'):
            try:
                idx, stem, descs = fut.result()
                raw_descs[idx] = (stem, descs)
            except Exception:
                pass

    print('  Phase 2: VLAD encoding with FAISS GPU assignment...')
    for idx, item in enumerate(tqdm(raw_descs, desc='VLAD (GPU assign)')):
        if item is None: continue
        stem, descs = item
        stms[idx] = stem
        if descs is not None and len(descs) > 0:
            mat[idx] = vlad_encode_faiss(descs, centroids, centroid_index)

    print(f'  Done in {time.time()-t0:.1f}s')
    if cache_vlad:
        np.save(cache_vlad, mat)
        with open(cache_stems,'wb') as f: pickle.dump(stms, f)
        print(f'  Cached → {cache_vlad.name}')
    return mat, stms


VLAD_RAW, DB_STEMS = build_database(
    ALL_IMG_PATHS, CENTROIDS, CENTROID_INDEX,
    cache_vlad=VLAD_RAW_CACHE, cache_stems=STEMS_CACHE)
STEM_TO_IDX = {s:i for i,s in enumerate(DB_STEMS)}
print(f'VLAD_RAW: {VLAD_RAW.shape}')

[CACHE] DB ← vlad_raw_K1024.npy
  11475 vectors, dim=131072
VLAD_RAW: (11475, 131072)


## Offline Step 5 — PCA + Whitening (GPU if cuML available, else CPU)

In [11]:
PCA_CACHE  = CACHE_DIR / f'pca_K{K_VLAD}_D{PCA_DIM}.pkl'
DBIDX_PATH = CACHE_DIR / f'db_index_K{K_VLAD}_D{PCA_DIM}.npy'

def fit_pca_whitening(vlad_matrix, n_components=PCA_DIM, cache_path=None):
    if cache_path and cache_path.exists():
        print(f'[CACHE] PCA ← {cache_path.name}')
        with open(cache_path,'rb') as f: return pickle.load(f)

    t0 = time.time()
    if USE_CUML:
        print(f'[OFFLINE] PCA via cuML GPU (n={n_components})...')
        import cupy as cp
        pca = cuPCA(n_components=n_components, whiten=True)
        pca.fit(cp.array(vlad_matrix))
    else:
        print(f'[OFFLINE] PCA via sklearn CPU (n={n_components})...')
        pca = PCA(n_components=n_components, whiten=True, random_state=42)
        pca.fit(vlad_matrix)

    ratio = pca.explained_variance_ratio_.sum()
    if hasattr(ratio, 'get'): ratio = ratio.get()  # cupy → numpy
    print(f'  Explained variance: {ratio*100:.1f}%  ({time.time()-t0:.1f}s)')

    if cache_path:
        with open(cache_path,'wb') as f: pickle.dump(pca, f)
        print(f'  Cached → {cache_path.name}')
    return pca


def apply_pca_whitening(vlad_matrix, pca):
    if USE_CUML and hasattr(pca, 'transform'):
        try:
            import cupy as cp
            proj = pca.transform(cp.array(vlad_matrix))
            proj = cp.asnumpy(proj) if hasattr(proj, 'get') else np.array(proj)
        except Exception:
            proj = pca.transform(vlad_matrix)
    else:
        proj = pca.transform(vlad_matrix)

    if hasattr(proj, 'get'): proj = proj.get()
    return normalize(proj.astype(np.float32), norm='l2')


PCA_MODEL = fit_pca_whitening(VLAD_RAW, n_components=PCA_DIM, cache_path=PCA_CACHE)

if DBIDX_PATH.exists():
    print(f'[CACHE] DB index ← {DBIDX_PATH.name}')
    DB_VLAD = np.load(DBIDX_PATH)
else:
    print('[OFFLINE] Projecting DB vectors...')
    DB_VLAD = apply_pca_whitening(VLAD_RAW, PCA_MODEL)
    np.save(DBIDX_PATH, DB_VLAD)

print(f'DB index: {DB_VLAD.shape}  norm_mean={np.linalg.norm(DB_VLAD,axis=1).mean():.4f}')
print('\n✅  OFFLINE STAGE COMPLETE')

[CACHE] PCA ← pca_K1024_D256.pkl
[CACHE] DB index ← db_index_K1024_D256.npy
DB index: (11475, 256)  norm_mean=1.0000

✅  OFFLINE STAGE COMPLETE


---
# 🟢 ONLINE STAGE
## Online Step 1 — Build FAISS Retrieval Index (GPU)
> `IndexFlatIP` = exact inner product search (cosine sim on L2-normalized vectors).  
> On GPU this is **50-100× faster** than `DB_VLAD @ query_vec` in NumPy for large DBs.

In [12]:
def build_retrieval_index(db_vlad):
    """
    Build a FAISS IndexFlatIP over the final DB vectors.
    Inner product on L2-normalized vectors = cosine similarity.
    GPU version gives near-instant retrieval even for 10k+ images.
    """
    index_cpu = faiss.IndexFlatIP(db_vlad.shape[1])   # inner product
    index_cpu.add(db_vlad)
    if USE_FAISS_GPU:
        index = faiss.index_cpu_to_gpu(GPU_RES, 0, index_cpu)
        print(f'Retrieval index on GPU  ({db_vlad.shape[0]} vectors, dim={db_vlad.shape[1]})')
    else:
        index = index_cpu
        print(f'Retrieval index on CPU  ({db_vlad.shape[0]} vectors, dim={db_vlad.shape[1]})')
    return index


RETRIEVAL_INDEX = build_retrieval_index(DB_VLAD)


def encode_query(image_path, roi=None):
    """RootSIFT (inside ROI) → VLAD (GPU assign) → PCA+whiten → L2 norm."""
    _, descs = extract_rootsift(image_path, roi=roi)
    if descs is None or len(descs) == 0:
        return np.zeros(PCA_DIM, dtype=np.float32)
    vlad = vlad_encode_faiss(descs, CENTROIDS, CENTROID_INDEX)
    return apply_pca_whitening(vlad[np.newaxis], PCA_MODEL)[0]


def initial_retrieve(query_vec, exclude=None, top_k=None):
    """
    GPU-accelerated retrieval via FAISS IndexFlatIP.
    Returns (ranked_stems, scores).
    """
    n_return = N_IMAGES  # retrieve all and sort
    q = np.ascontiguousarray(query_vec[np.newaxis].astype(np.float32))  # (1, D)
    scores, indices = RETRIEVAL_INDEX.search(q, n_return)               # (1, N)
    scores  = scores[0]
    indices = indices[0]

    stems  = [DB_STEMS[i] for i in indices]
    if exclude:
        mask   = [s != exclude for s in stems]
        stems  = [s for s,m in zip(stems,mask) if m]
        scores = scores[np.array(mask)]

    return stems, scores


print('GPU retrieval index ready.')

Retrieval index on CPU  (11475 vectors, dim=256)
GPU retrieval index ready.


## Online Step 2 — RANSAC Spatial Re-ranking (CPU)
> OpenCV SIFT + RANSAC is CPU-bound. We keep this on CPU but it only runs on top-100 candidates.

In [13]:
def ransac_rerank(query_path, query_roi, ranked_stems,
                  top_n=RERANK_TOP_N, min_inliers=MIN_INLIERS):
    to_rerank = ranked_stems[:top_n]
    rest      = ranked_stems[top_n:]
    q_kps, q_descs = extract_rootsift(query_path, roi=query_roi)
    if q_descs is None or len(q_kps) < 4: return ranked_stems
    q_pts = np.float32([kp.pt for kp in q_kps])
    flann = cv2.FlannBasedMatcher(dict(algorithm=1,trees=5),dict(checks=50))
    inliers = {}
    for stem in to_rerank:
        path = STEM2PATH.get(stem)
        if path is None: inliers[stem]=0; continue
        c_kps, c_descs = extract_rootsift(path)
        if c_descs is None or len(c_kps)<4: inliers[stem]=0; continue
        c_pts = np.float32([kp.pt for kp in c_kps])
        try:
            matches = flann.knnMatch(q_descs, c_descs, k=2)
            good    = [m for p in matches if len(p)==2
                       for m,n in [p] if m.distance<0.75*n.distance]
            if len(good)<4: inliers[stem]=0; continue
            src = np.float32([q_pts[m.queryIdx] for m in good]).reshape(-1,1,2)
            dst = np.float32([c_pts[m.trainIdx] for m in good]).reshape(-1,1,2)
            _,mask = cv2.findHomography(src,dst,cv2.RANSAC,ransacReprojThreshold=10.0)
            inliers[stem] = int(mask.sum()) if mask is not None else 0
        except Exception: inliers[stem]=0
    def key(s):
        c = inliers.get(s,0)
        return (0 if c>=min_inliers else 1, -c)
    return sorted(to_rerank, key=key) + list(rest)

print('RANSAC re-ranker ready.')

RANSAC re-ranker ready.


## Online Step 3 — Average Query Expansion (GPU dot products)

In [14]:
def average_query_expansion(query_vec, ranked_stems, top_k=AQE_TOP_K):
    vecs = [query_vec]
    for stem in ranked_stems[:top_k]:
        idx = STEM_TO_IDX.get(stem)
        if idx is not None: vecs.append(DB_VLAD[idx])
    if len(vecs)==1: return query_vec
    exp = np.mean(vecs, axis=0).astype(np.float32)
    n   = np.linalg.norm(exp)
    return exp/n if n>1e-8 else exp


def _ap(ranked, good, ok, junk):
    pos = good|ok
    if not pos: return 0.0
    ap=n_ret=n_rel=0
    for s in ranked:
        if s in junk: continue
        n_ret+=1
        if s in pos: n_rel+=1; ap+=n_rel/n_ret
    return ap/len(pos)

def _pk(ranked, good, ok, junk, k=5):
    pos=good|ok; n_seen=n_rel=0
    for s in ranked:
        if s in junk: continue
        n_seen+=1
        if s in pos: n_rel+=1
        if n_seen==k: break
    return n_rel/k if k else 0.0


def run_query(query_info, use_ransac=True, use_aqe=True):
    q_path = STEM2PATH.get(query_info['query_img'])
    if q_path is None: return [], 0.0, 0.0
    q_roi = query_info['query_roi']
    good, ok, junk = query_info['good'], query_info['ok'], query_info['junk']

    q_vec          = encode_query(q_path, roi=q_roi)
    ranked, scores = initial_retrieve(q_vec, exclude=query_info['query_img'])

    if use_ransac:
        ranked = ransac_rerank(q_path, q_roi, ranked)
    if use_aqe:
        exp_q  = average_query_expansion(q_vec, ranked)
        ranked, _ = initial_retrieve(exp_q, exclude=query_info['query_img'])

    return ranked, _ap(ranked,good,ok,junk), _pk(ranked,good,ok,junk)

print('Full online pipeline ready.')

Full online pipeline ready.


---
# 📊 EVALUATION

In [15]:
def evaluate(queries, use_ransac=True, use_aqe=True, tag=''):
    all_ap, all_p5, results = [], [], {}
    for q in tqdm(queries, desc=tag or 'Eval'):
        ranked, ap, p5 = run_query(q, use_ransac, use_aqe)
        all_ap.append(ap); all_p5.append(p5)
        results[q['name']] = {'ranked':ranked,'ap':ap,'p5':p5,'query_info':q}
    mAP=float(np.mean(all_ap)); mP5=float(np.mean(all_p5))
    print(f'  {tag:42s} mAP={mAP:.4f}  P@5={mP5:.4f}')
    return mAP, mP5, results


print('='*55,'\nBASELINE\n'+'='*55)
mAP_ox_b=mAP_pa_b=0.0; res_ox_b=res_pa_b={}
if oxford_queries: mAP_ox_b,_,res_ox_b = evaluate(oxford_queries,False,False,'Oxford-Baseline')
if paris_queries:  mAP_pa_b,_,res_pa_b = evaluate(paris_queries, False,False,'Paris-Baseline')

BASELINE


Oxford-Baseline:   0%|          | 0/110 [00:00<?, ?it/s]

  Oxford-Baseline                            mAP=0.5162  P@5=0.8891


Paris-Baseline:   0%|          | 0/110 [00:00<?, ?it/s]

  Paris-Baseline                             mAP=0.2850  P@5=0.4982


In [16]:
print('='*55,'\nFULL PIPELINE (RANSAC + AQE)\n'+'='*55)
mAP_ox_f=mAP_pa_f=0.0; res_ox_f=res_pa_f={}
if oxford_queries: mAP_ox_f,_,res_ox_f = evaluate(oxford_queries,True,True,'Oxford-Full')
if paris_queries:  mAP_pa_f,_,res_pa_f = evaluate(paris_queries, True,True,'Paris-Full')

FULL PIPELINE (RANSAC + AQE)


Oxford-Full:   0%|          | 0/110 [00:00<?, ?it/s]

  Oxford-Full                                mAP=0.6413  P@5=0.9455


Paris-Full:   0%|          | 0/110 [00:00<?, ?it/s]

KeyboardInterrupt: 

---
# 📊 ABLATION STUDY

In [ ]:
ablation = {}
eval_qs  = oxford_queries or paris_queries
eval_tag = 'Oxford' if oxford_queries else 'Paris'
for label,r,a in [
    ('Baseline (VLAD+PCA-Wh)', False, False),
    ('+ RANSAC only',          True,  False),
    ('+ AQE only',             False, True),
    ('+ RANSAC + AQE',         True,  True),
]:
    v,_,_ = evaluate(eval_qs,r,a,label); ablation[label]=v

print(f'\nAblation ({eval_tag}):')
for k,v in ablation.items(): print(f'  {k:35s}: {v:.4f}')

---
# 📊 VOCABULARY SIZE EXPERIMENT

In [ ]:
vocab_map = {}; eval_qs_sub = eval_qs[:20]
for k in [64, 128, 256, 512]:
    v_c=CACHE_DIR/f'vocab_K{k}.npy'; d_c=CACHE_DIR/f'vlad_raw_K{k}.npy'
    s_c=CACHE_DIR/f'db_stems_K{k}.pkl'; p_c=CACHE_DIR/f'pca_K{k}_D{PCA_DIM}.pkl'

    ck = build_vocabulary_faiss(ALL_IMG_PATHS,k=k,n_sample_imgs=1500,cache_path=v_c)
    ci = build_centroid_index(ck)

    # Use build_database with the new centroid index
    _old_ci = CENTROID_INDEX
    globals()['CENTROID_INDEX'] = ci
    vr,sk = build_database(ALL_IMG_PATHS,ck,ci,cache_vlad=d_c,cache_stems=s_c)
    globals()['CENTROID_INDEX'] = _old_ci

    pk = fit_pca_whitening(vr,n_components=min(PCA_DIM,k*128),cache_path=p_c)
    dk = apply_pca_whitening(vr,pk)
    ri = build_retrieval_index(dk)

    _bak = dict(DB_VLAD=DB_VLAD,DB_STEMS=list(DB_STEMS),STEM_TO_IDX=dict(STEM_TO_IDX),
                CENTROIDS=CENTROIDS.copy(),PCA_MODEL=PCA_MODEL,
                CENTROID_INDEX=CENTROID_INDEX,RETRIEVAL_INDEX=RETRIEVAL_INDEX)
    globals().update(dict(DB_VLAD=dk,DB_STEMS=sk,STEM_TO_IDX={s:i for i,s in enumerate(sk)},
                          CENTROIDS=ck,PCA_MODEL=pk,CENTROID_INDEX=ci,RETRIEVAL_INDEX=ri))

    mAP_k,_,_ = evaluate(eval_qs_sub,False,False,f'K={k}'); vocab_map[k]=mAP_k
    globals().update(_bak)

print('Vocab experiment:', vocab_map)

---
# 🖼️ QUALITATIVE EVALUATION

In [ ]:
def show_top5(query_info, ranked_stems, n_show=5):
    pos=query_info['good']|query_info['ok']; junk=query_info['junk']
    top5=[s for s in ranked_stems if s not in junk][:n_show]
    fig,axes=plt.subplots(1,n_show+1,figsize=(4*(n_show+1),4))
    qname=query_info['name'].replace('oxford_','').replace('paris_','')
    def load(stem):
        p=STEM2PATH.get(stem)
        if p is None: return np.zeros((200,200,3),dtype=np.uint8)
        img=cv2.imread(str(p))
        return cv2.cvtColor(img,cv2.COLOR_BGR2RGB) if img is not None else np.zeros((200,200,3),dtype=np.uint8)
    ax=axes[0]; ax.imshow(load(query_info['query_img']))
    if query_info['query_roi']:
        x1,y1,x2,y2=query_info['query_roi']
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=3,edgecolor='yellow',facecolor='none'))
    ax.set_title(f'QUERY\n{qname}',fontsize=9,fontweight='bold'); ax.axis('off')
    for i,stem in enumerate(top5):
        ax=axes[i+1]; ax.imshow(load(stem))
        rel=stem in pos; color='limegreen' if rel else 'tomato'
        ax.set_title(f'#{i+1} {"✓" if rel else "✗"}',color=color,fontsize=11,fontweight='bold')
        for sp in ax.spines.values(): sp.set_edgecolor(color); sp.set_linewidth(5)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR/f'query_{qname}.png',dpi=90,bbox_inches='tight')
    plt.show()

res_main=res_ox_f if oxford_queries else res_pa_f
qs_main =oxford_queries if oxford_queries else paris_queries
sorted_qs=sorted([q for q in qs_main if q['name'] in res_main],
                  key=lambda q:res_main[q['name']]['ap'],reverse=True)
for label,q in [('BEST',sorted_qs[0]),('MEDIAN',sorted_qs[len(sorted_qs)//2]),('WORST',sorted_qs[-1])]:
    r=res_main[q['name']]; print(f'\n── {label} — AP={r["ap"]:.3f} ──')
    show_top5(q,r['ranked'])

---
# 📊 RESULTS DASHBOARD

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(18,12))
fig.suptitle('Image Retrieval — Results Dashboard',fontsize=17,fontweight='bold')
res_main=res_ox_f if oxford_queries else res_pa_f
ds_label='Oxford' if oxford_queries else 'Paris'
names=[k.replace('oxford_','').replace('paris_','') for k in res_main]
aps=[res_main[k]['ap'] for k in res_main]; med=float(np.median(aps))

ax=axes[0,0]
ax.barh(range(len(names)),aps,color=['steelblue' if a>=med else 'tomato' for a in aps],edgecolor='white',height=0.7)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names,fontsize=7)
ax.axvline(np.mean(aps),color='k',ls='--',lw=1.5,label=f'mAP={np.mean(aps):.3f}')
ax.legend(); ax.set_xlabel('AP'); ax.set_title(f'{ds_label} Per-Query AP'); ax.set_xlim(0,1)

ax=axes[0,1]
labs,vals=list(ablation.keys()),list(ablation.values())
bars=ax.barh(labs,vals,color=['#4C72B0','#DD8452','#55A868','#C44E52'],edgecolor='white')
for bar,v in zip(bars,vals): ax.text(v+0.005,bar.get_y()+bar.get_height()/2,f'{v:.4f}',va='center',fontsize=9)
ax.set_xlabel('mAP'); ax.set_title(f'{eval_tag} Ablation'); ax.set_xlim(0,max(vals)*1.15)

ax=axes[1,0]
if vocab_map:
    ks,ms=list(vocab_map.keys()),list(vocab_map.values())
    ax.plot(ks,ms,'o-',color='steelblue',lw=2,ms=9)
    for k,m in zip(ks,ms): ax.annotate(f'{m:.3f}',(k,m),xytext=(0,10),textcoords='offset points',ha='center',fontsize=9)
    ax.set_xlabel('K'); ax.set_ylabel('mAP'); ax.set_title('Vocabulary Size vs mAP')
    ax.set_xscale('log',base=2); ax.set_xticks(ks); ax.set_xticklabels(ks); ax.grid(True,alpha=0.3)

ax=axes[1,1]
ax.hist(aps,bins=12,color='steelblue',edgecolor='white',alpha=0.85)
ax.axvline(np.mean(aps),color='red',ls='--',lw=1.5,label=f'Mean={np.mean(aps):.3f}')
ax.axvline(np.median(aps),color='orange',ls=':',lw=1.5,label=f'Median={np.median(aps):.3f}')
ax.set_xlabel('AP'); ax.set_ylabel('# Queries'); ax.set_title('AP Distribution'); ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR/'results_dashboard.png',dpi=120,bbox_inches='tight')
plt.show()

---
# 📋 FINAL SUMMARY

In [ ]:
W=62
def row(l,v): print(f'║  {l:<30}: {str(v):<{W-36}}║')
print('╔'+'═'*(W-2)+'╗')
print('║'+'  HANDCRAFTED IMAGE RETRIEVAL — SUMMARY'.center(W-2)+'║')
print('╠'+'═'*(W-2)+'╣')
row('Total DB images', N_IMAGES)
row('Oxford queries', len(oxford_queries))
row('Paris  queries', len(paris_queries))
row('K_VLAD', K_VLAD)
row('PCA dim', PCA_DIM)
row('GPU acceleration', f'FAISS-{"GPU" if USE_FAISS_GPU else "CPU"}, CuPy={USE_CUPY}, cuML={USE_CUML}')
print('╠'+'═'*(W-2)+'╣')
if oxford_queries:
    row('Oxford baseline mAP', f'{mAP_ox_b:.4f}')
    row('Oxford full     mAP', f'{mAP_ox_f:.4f}  (+{(mAP_ox_f-mAP_ox_b)*100:.1f}%)')
if paris_queries:
    row('Paris  baseline mAP', f'{mAP_pa_b:.4f}')
    row('Paris  full     mAP', f'{mAP_pa_f:.4f}  (+{(mAP_pa_f-mAP_pa_b)*100:.1f}%)')
print('╠'+'═'*(W-2)+'╣')
for l,v in ablation.items(): row(l[:30],f'{v:.4f}')
if vocab_map:
    best_k=max(vocab_map,key=vocab_map.get)
    print('╠'+'═'*(W-2)+'╣')
    row('Best K',f'K={best_k}  mAP={vocab_map[best_k]:.4f}')
print('╚'+'═'*(W-2)+'╝')